# Real Azure Traffic Traces for Admission Control

This notebook demonstrates the dataset-construction pipeline behind **`data.py`**: it turns real-world serverless-function request traces (from the Azure Functions 2019 invocation/duration trace, Shahrad et al., USENIX ATC 2020) into a schema-standardized, request-level dataset for evaluating **admission-control policies under overloaded request queues**.

Each row is one incoming request. `input` carries only information available **at admission time** (arrival time, an imperfect risk heuristic, the SLO target, and regime/function identifiers). `output` is the **SLO-violation label**, computed *post-hoc* from the request's realized service time — information that is deliberately excluded from `input` to avoid label leakage.

The real dataset spans 5 traffic regimes (`stationary`, `burst`, `drift`, `regime_switch`, `adversarial`); only `adversarial` is synthetically constructed, and it is explicitly flagged via `is_synthetic`/`provenance` fields.

This demo runs the **exact same** `subsample_rows` / `build_example` logic as the original `data.py`, on a small stratified sample of raw request rows (20 per regime) instead of the full ~540K-row trace, so it finishes in seconds while producing genuinely the same output structure.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# loguru -- not pre-installed on Colab, always install
_pip('loguru==0.7.3')

# numpy -- pre-installed on Colab, install locally only (to match Colab's ABI)
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2')

In [ ]:
# --- Imports (copied from data.py, plus matplotlib for the results visualization) ---
import random
import json
import sys
from pathlib import Path

from loguru import logger
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

## Load the data

`mini_demo_data.json` is a stratified sample of **raw request rows** (20 per traffic regime, 100 total) drawn directly from the real `raw_azure_admission_control.json` trace that `data.py` normally reads — same `schema_doc`/`provenance_summary`/`requests` structure, just far fewer rows, so the exact same downstream code runs unmodified.

The loader tries the GitHub raw URL first (so this notebook works standalone on Colab), then falls back to a local copy of the file (so it works when run from this repo checkout).

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-papers/ai-invention-db8806-conformal-admission-control-distribution/main/round-1/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
requests = data["requests"]
print(f"Loaded {len(requests)} raw request rows")
print("Regimes present:", sorted({r['regime_label'] for r in requests}))
requests[0]

## Config

The original `data.py` caps each regime at a few tens of thousands of rows (`REGIME_CAPS`) so the 210K-example full dataset stays under the file-size limit. Since our demo sample already has only 20 rows per regime, we set the caps to the **absolute minimum that keeps every row** (i.e. no-op subsampling here) — bump these back up to the original values (commented) to reproduce the full-scale dataset if you have the full `raw_azure_admission_control.json` trace locally.

In [ ]:
# Original (full-scale) values -- uncomment to reproduce the real 210K-example dataset
# given the full raw_azure_admission_control.json trace:
# REGIME_CAPS = {
#     "stationary": 50000,
#     "burst": 40000,
#     "drift": 50000,
#     "regime_switch": 50000,
#     "adversarial": 20000,
# }
# N_PARTS = 4  # keeps each split part comfortably under the 100MB file size limit

# Demo (minimum) values: keep every row in our 100-row sample -- 20/regime, well under any cap
REGIME_CAPS = {
    "stationary": 20,
    "burst": 20,
    "drift": 20,
    "regime_switch": 20,
    "adversarial": 20,
}
N_PARTS = 1  # a sample this small fits in a single part
SEED = 20260825

## Stratified per-regime subsampling

`subsample_rows` (copied verbatim from `data.py`) groups raw requests by `regime_label` and samples down to `REGIME_CAPS[regime]` per regime, while preserving each regime's own `arrival_time` ordering (sampled indices are sorted before being used to index back into `rows`).

In [ ]:
def subsample_rows(requests: list[dict]) -> list[dict]:
    """Stratified subsample per regime, capped per REGIME_CAPS, to stay well
    under the 300MB output limit while keeping every regime's >=2000-decision
    floor and preserving each regime's own arrival_time ordering."""
    by_regime: dict[str, list[dict]] = {}
    for r in requests:
        by_regime.setdefault(r["regime_label"], []).append(r)

    rng = random.Random(SEED)
    kept: list[dict] = []
    for regime, rows in by_regime.items():
        cap = REGIME_CAPS.get(regime, len(rows))
        if len(rows) <= cap:
            kept.extend(rows)
            continue
        sampled_idx = sorted(rng.sample(range(len(rows)), cap))
        kept.extend(rows[i] for i in sampled_idx)
    return kept

## Building admission-control examples

`build_example` (copied verbatim from `data.py`) converts one raw request row into the `exp_sel_data_out.json` example format:

- `input` = a JSON string of **admission-time-only** features (`arrival_time`, `risk_score`, `slo_target`, `regime_label`, `function_id`, `is_synthetic`).
- `output` = `"1"` iff `service_time > slo_target` (an SLO violation), else `"0"` -- computed from `service_time`, which is deliberately **not** included in `input`.
- `metadata_*` fields carry everything else (fold, task type, regime, provenance, the realized `service_time`, etc.) for downstream inspection without leaking into the model-visible `input`.

In [ ]:
FOLD_TO_INT = {"train": 0, "val": 1, "test": 2}


def build_example(row: dict) -> dict:
    is_violation = row["service_time"] > row["slo_target"]
    input_features = {
        "arrival_time": row["arrival_time"],
        "risk_score": row["risk_score"],
        "slo_target": row["slo_target"],
        "regime_label": row["regime_label"],
        "function_id": row["function_id"],
        "is_synthetic": row["is_synthetic"],
    }
    return {
        "input": json.dumps(input_features),
        "output": "1" if is_violation else "0",
        "metadata_fold": FOLD_TO_INT[row["metadata_fold"]],
        "metadata_task_type": "classification",
        "metadata_n_classes": 2,
        "metadata_regime_label": row["regime_label"],
        "metadata_function_id": row["function_id"],
        "metadata_request_id": row["request_id"],
        "metadata_is_synthetic": row["is_synthetic"],
        "metadata_provenance": row["provenance"],
        "metadata_service_time": row["service_time"],
        "metadata_slo_target": row["slo_target"],
        "metadata_feature_names": list(input_features.keys()),
    }

## Running the pipeline (adapted from `data.py`'s `main()`)

Same logic as the original `main()`: subsample, convert every row to an example, log the per-example failures (if any), then compute the overall and per-regime SLO-violation rates. The only changes from the original are dropping the file-writing/part-splitting step (not needed for a 100-row in-memory demo) and using the small `REGIME_CAPS`/`N_PARTS` from the config cell above instead of the full-scale constants.

In [ ]:
logger.info(f"Loaded {len(requests)} raw request rows")

requests_sub = subsample_rows(requests)
logger.info(f"Subsampled to {len(requests_sub)} rows (per-regime caps={REGIME_CAPS})")

examples = []
for i, row in enumerate(requests_sub):
    try:
        examples.append(build_example(row))
    except (KeyError, TypeError) as e:
        logger.error(f"Failed to convert row {i}: {e}")
        continue

logger.info(f"Converted {len(examples)}/{len(requests_sub)} rows to examples")

n_violations = sum(1 for e in examples if e["output"] == "1")
logger.info(f"Overall violation rate: {n_violations / len(examples):.4f}")
by_regime: dict[str, list[int]] = {}
for e in examples:
    by_regime.setdefault(e["metadata_regime_label"], []).append(1 if e["output"] == "1" else 0)
for regime, labels in by_regime.items():
    logger.info(f"  regime={regime}: n={len(labels)} violation_rate={sum(labels) / len(labels):.4f}")

## Results

Each example's `risk_score` is a deliberately imperfect, admission-time-only heuristic -- the hypothesis is that it carries weak but real signal about SLO violation risk, unevenly so across regimes. Below we show one example per regime, the per-regime violation rate, and a scatter of `risk_score` vs. the (post-hoc) violation label to visualize how weak/miscalibrated that admission-time signal actually is on this sample.

In [ ]:
print(f"{'regime':<15}{'n':>5}{'violation_rate':>16}")
regimes_sorted = sorted(by_regime)
for regime in regimes_sorted:
    labels = by_regime[regime]
    print(f"{regime:<15}{len(labels):>5}{sum(labels) / len(labels):>16.4f}")

print()
print("One example per regime (input, output):")
seen = set()
for e in examples:
    r = e["metadata_regime_label"]
    if r in seen:
        continue
    seen.add(r)
    print(f"  [{r}] input={e['input']}  ->  output={e['output']}")

risk_scores = [json.loads(e["input"])["risk_score"] for e in examples]
labels = [int(e["output"]) for e in examples]
regime_of = [e["metadata_regime_label"] for e in examples]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# left: violation rate per regime
rates = [sum(by_regime[r]) / len(by_regime[r]) for r in regimes_sorted]
axes[0].bar(regimes_sorted, rates, color="#4C72B0")
axes[0].set_ylabel("SLO-violation rate")
axes[0].set_title("Violation rate by regime (demo sample)")
axes[0].tick_params(axis="x", rotation=30)

# right: risk_score vs. violation label, colored by regime
colors = {r: c for r, c in zip(regimes_sorted, plt.cm.tab10.colors)}
for r in regimes_sorted:
    xs = [rs for rs, rg in zip(risk_scores, regime_of) if rg == r]
    ys = [lb + (0.02 * (hash(r) % 5 - 2)) for lb, rg in zip(labels, regime_of) if rg == r]  # jitter for visibility
    axes[1].scatter(xs, ys, label=r, color=colors[r], alpha=0.7)
axes[1].set_xlabel("risk_score (admission-time heuristic)")
axes[1].set_ylabel("SLO violation label (jittered)")
axes[1].set_title("Admission-time risk_score vs. realized violation")
axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))

plt.tight_layout()
plt.show()